## Aggregation dataframe

In [26]:
import pandas as pd

orders_data_path = '../../../resources/data/retail_db/orders/part-00000'
orders_columns = ['order_id', 'order_date', 'order_customer_id', 'order_status']

orders = pd.read_csv(orders_data_path, names=orders_columns)

orders.head()

,order_id,order_date,order_customer_id,order_status
0,1,2013-07-25 00:00:00.0,11599,CLOSED
1,2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
2,3,2013-07-25 00:00:00.0,12111,COMPLETE
3,4,2013-07-25 00:00:00.0,8827,CLOSED
4,5,2013-07-25 00:00:00.0,11318,COMPLETE


### Get count by order_id. using Pandas

In [27]:
orders.columns

Index(['order_id', 'order_date', 'order_customer_id', 'order_status'], dtype='str')

In [28]:
orders. \
    groupby('order_status').agg('count')

,order_id,order_date,order_customer_id
order_status,,,
CANCELED,1428,1428,1428
CLOSED,7556,7556,7556
COMPLETE,22899,22899,22899
ON_HOLD,3798,3798,3798
PAYMENT_REVIEW,729,729,729
PENDING,7610,7610,7610
PENDING_PAYMENT,15030,15030,15030
PROCESSING,8275,8275,8275
SUSPECTED_FRAUD,1558,1558,1558


In [29]:
orders. \
    groupby('order_status')['order_id']. \
    agg('count')

order_status
CANCELED            1428
CLOSED              7556
COMPLETE           22899
ON_HOLD             3798
PAYMENT_REVIEW       729
PENDING             7610
PENDING_PAYMENT    15030
PROCESSING          8275
SUSPECTED_FRAUD     1558
Name: order_id, dtype: int64

### Get count by month

In [30]:
orders['order_month'] = orders.apply(lambda order: order.order_date[:7], axis=1)

orders.head()

,order_id,order_date,order_customer_id,order_status,order_month
0,1,2013-07-25 00:00:00.0,11599,CLOSED,2013-07
1,2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,2013-07
2,3,2013-07-25 00:00:00.0,12111,COMPLETE,2013-07
3,4,2013-07-25 00:00:00.0,8827,CLOSED,2013-07
4,5,2013-07-25 00:00:00.0,11318,COMPLETE,2013-07


In [31]:
orders. \
    groupby(['order_month', 'order_status'])['order_id']. \
    agg(order_count='count')

order_count
order_month order_status                
2013-07     CANCELED                  22
            CLOSED                   161
            COMPLETE                 515
            ON_HOLD                   81
            PAYMENT_REVIEW            19
...                                  ...
2014-07     PAYMENT_REVIEW            54
            PENDING                  517
            PENDING_PAYMENT          979
            PROCESSING               561
            SUSPECTED_FRAUD          101

[117 rows x 1 columns]

In [32]:
orders. \
    groupby(['order_month', 'order_status'])['order_id']. \
    agg(order_count='count'). \
    reset_index()

,order_month,order_status,order_count
0,2013-07,CANCELED,22
1,2013-07,CLOSED,161
2,2013-07,COMPLETE,515
3,2013-07,ON_HOLD,81
4,2013-07,PAYMENT_REVIEW,19
...,...,...,...
112,2014-07,PAYMENT_REVIEW,54
113,2014-07,PENDING,517
114,2014-07,PENDING_PAYMENT,979
115,2014-07,PROCESSING,561


## Joining Dataframe

In [33]:
import json
import pandas as pd

In [34]:
def get_column_names(schemas, ds_name, sorting_key='column_position'):
    columns_details = schemas[ds_name]
    columns = sorted(columns_details , key=lambda col: col[sorting_key])
    
    return [col['column_name'] for col in columns]

In [35]:
json_schema_path = '../../../resources/data/retail_db/schemas.json'

schemas = json.load(open(json_schema_path))

In [36]:
orders_columns = get_column_names(schemas, 'orders')

In [37]:
orders_data_path = '../../../resources/data/retail_db/orders/part-00000'
orders = pd.read_csv(orders_data_path, names=orders_columns)

orders.head()

,order_id,order_date,order_customer_id,order_status
0,1,2013-07-25 00:00:00.0,11599,CLOSED
1,2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
2,3,2013-07-25 00:00:00.0,12111,COMPLETE
3,4,2013-07-25 00:00:00.0,8827,CLOSED
4,5,2013-07-25 00:00:00.0,11318,COMPLETE


In [38]:
customers_columns = get_column_names(schemas, 'customers')
customers_columns

['customer_id',
 'customer_fname',
 'customer_lname',
 'customer_email',
 'customer_password',
 'customer_street',
 'customer_city',
 'customer_state',
 'customer_zipcode']

In [39]:
customers = pd.read_csv(orders_data_path, names=customers_columns)
customers

,customer_id,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode
0,1,2013-07-25 00:00:00.0,11599,CLOSED,NaN,NaN,NaN,NaN,NaN
1,2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,NaN,NaN,NaN,NaN,NaN
2,3,2013-07-25 00:00:00.0,12111,COMPLETE,NaN,NaN,NaN,NaN,NaN
3,4,2013-07-25 00:00:00.0,8827,CLOSED,NaN,NaN,NaN,NaN,NaN
4,5,2013-07-25 00:00:00.0,11318,COMPLETE,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
68878,68879,2014-07-09 00:00:00.0,778,COMPLETE,NaN,NaN,NaN,NaN,NaN
68879,68880,2014-07-13 00:00:00.0,1117,COMPLETE,NaN,NaN,NaN,NaN,NaN
68880,68881,2014-07-19 00:00:00.0,2518,PENDING_PAYMENT,NaN,NaN,NaN,NaN,NaN
68881,68882,2014-07-22 00:00:00.0,10000,ON_HOLD,NaN,NaN,NaN,NaN,NaN


### Inner join

In [40]:
orders = orders.set_index('order_customer_id')

In [41]:
customers = customers.set_index('customer_id')

In [42]:
customers_orders = customers.join(
    orders, how='inner'
)

customers_orders.head()

,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode,order_id,order_date,order_status
customer_id,,,,,,,,,,,
10829,2013-09-29 00:00:00.0,2925,CLOSED,NaN,NaN,NaN,NaN,NaN,1745,2013-08-03 00:00:00.0,COMPLETE
5763,2013-08-29 00:00:00.0,3433,COMPLETE,NaN,NaN,NaN,NaN,NaN,1424,2013-08-01 00:00:00.0,COMPLETE
11656,2013-10-04 00:00:00.0,11181,PENDING,NaN,NaN,NaN,NaN,NaN,58116,2013-08-08 00:00:00.0,PENDING
11150,2013-10-02 00:00:00.0,11325,CLOSED,NaN,NaN,NaN,NaN,NaN,18299,2013-11-14 00:00:00.0,PENDING_PAYMENT
3119,2013-08-11 00:00:00.0,9773,PENDING,NaN,NaN,NaN,NaN,NaN,55846,2014-07-13 00:00:00.0,PAYMENT_REVIEW


In [43]:
customers_orders.shape

(68883, 11)

### Perform Aggregation on Join results

In [44]:
customers_orders. \
    reset_index(names='customer_id'). \
        groupby('customer_id')['customer_id']. \
            agg(order_count='count'). \
                reset_index(). \
                    query(' order_count >= 10 ')

,customer_id,order_count
70,71,10
171,172,10
173,174,12
196,197,11
219,221,15
...,...,...
12311,12341,10
12317,12347,10
12375,12406,10
12400,12431,16


In [45]:
customers_orders

,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode,order_id,order_date,order_status
customer_id,,,,,,,,,,,
10829,2013-09-29 00:00:00.0,2925,CLOSED,NaN,NaN,NaN,NaN,NaN,1745,2013-08-03 00:00:00.0,COMPLETE
5763,2013-08-29 00:00:00.0,3433,COMPLETE,NaN,NaN,NaN,NaN,NaN,1424,2013-08-01 00:00:00.0,COMPLETE
11656,2013-10-04 00:00:00.0,11181,PENDING,NaN,NaN,NaN,NaN,NaN,58116,2013-08-08 00:00:00.0,PENDING
11150,2013-10-02 00:00:00.0,11325,CLOSED,NaN,NaN,NaN,NaN,NaN,18299,2013-11-14 00:00:00.0,PENDING_PAYMENT
3119,2013-08-11 00:00:00.0,9773,PENDING,NaN,NaN,NaN,NaN,NaN,55846,2014-07-13 00:00:00.0,PAYMENT_REVIEW
...,...,...,...,...,...,...,...,...,...,...,...
4294,2013-08-19 00:00:00.0,5549,PAYMENT_REVIEW,NaN,NaN,NaN,NaN,NaN,67627,2013-09-16 00:00:00.0,PENDING_PAYMENT
781,2013-07-29 00:00:00.0,7971,PENDING,NaN,NaN,NaN,NaN,NaN,68026,2014-01-18 00:00:00.0,PROCESSING
10605,2013-09-28 00:00:00.0,618,COMPLETE,NaN,NaN,NaN,NaN,NaN,68544,2014-06-12 00:00:00.0,PROCESSING


## Sorting dataframe

In [46]:
orders = orders.reset_index()

In [47]:
orders.head()

,order_customer_id,order_id,order_date,order_status
0,11599,1,2013-07-25 00:00:00.0,CLOSED
1,256,2,2013-07-25 00:00:00.0,PENDING_PAYMENT
2,12111,3,2013-07-25 00:00:00.0,COMPLETE
3,8827,4,2013-07-25 00:00:00.0,CLOSED
4,11318,5,2013-07-25 00:00:00.0,COMPLETE


In [48]:
# sort orders df by order_customer_id descending order 
orders.sort_values('order_customer_id', ascending=False)

,order_customer_id,order_id,order_date,order_status
41642,12435,41643,2014-04-08 00:00:00.0,PENDING
61628,12435,61629,2013-12-21 00:00:00.0,CANCELED
6159,12434,6160,2013-09-02 00:00:00.0,COMPLETE
4798,12434,4799,2013-08-23 00:00:00.0,PENDING_PAYMENT
5302,12434,5303,2013-08-26 00:00:00.0,PENDING
...,...,...,...,...
57962,2,57963,2013-08-02 00:00:00.0,ON_HOLD
33864,2,33865,2014-02-18 00:00:00.0,COMPLETE
15191,2,15192,2013-10-29 00:00:00.0,PENDING_PAYMENT
67862,2,67863,2013-11-30 00:00:00.0,COMPLETE


In [49]:
# sort order_customer_id by ascending order
# sort order_date by descending order
orders.sort_values(
    ['order_customer_id', 'order_date'], ascending=[True, False]
)

,order_customer_id,order_id,order_date,order_status
22944,1,22945,2013-12-13 00:00:00.0,COMPLETE
33864,2,33865,2014-02-18 00:00:00.0,COMPLETE
67862,2,67863,2013-11-30 00:00:00.0,COMPLETE
15191,2,15192,2013-10-29 00:00:00.0,PENDING_PAYMENT
57962,2,57963,2013-08-02 00:00:00.0,ON_HOLD
...,...,...,...,...
5302,12434,5303,2013-08-26 00:00:00.0,PENDING
4798,12434,4799,2013-08-23 00:00:00.0,PENDING_PAYMENT
1867,12434,1868,2013-08-03 00:00:00.0,CLOSED
41642,12435,41643,2014-04-08 00:00:00.0,PENDING


## Save to CSV

In [50]:
save_file_path = '../../../resources/data/retail_db/orders/part-00000'

In [51]:
# orders.to_csv(save_file_path)

## Save to JSON

In [53]:
import os

os.makedirs('../../../resources/data/retail_db/orders_json', exist_ok=True)

In [55]:
orders.to_json(
    '../../../resources/data/retail_db/orders_json/part-00000.json',
    orient='records',
    lines=True    
)

In [56]:
pd.read_json(
    '../../../resources/data/retail_db/orders_json/part-00000.json',
    lines=True
)

,order_customer_id,order_id,order_date,order_status
0,11599,1,2013-07-25 00:00:00.0,CLOSED
1,256,2,2013-07-25 00:00:00.0,PENDING_PAYMENT
2,12111,3,2013-07-25 00:00:00.0,COMPLETE
3,8827,4,2013-07-25 00:00:00.0,CLOSED
4,11318,5,2013-07-25 00:00:00.0,COMPLETE
...,...,...,...,...
68878,778,68879,2014-07-09 00:00:00.0,COMPLETE
68879,1117,68880,2014-07-13 00:00:00.0,COMPLETE
68880,2518,68881,2014-07-19 00:00:00.0,PENDING_PAYMENT
68881,10000,68882,2014-07-22 00:00:00.0,ON_HOLD
